In [6]:
import random
import os
import json
from openai import OpenAI


message_template="""
Task Description:
Extract quantity relations from the given text and output the results in a specific TSV format. Use a Chain of Thought reasoning process to ensure the extraction is accurate and logical.

Output Format:
Provide results strictly in the following TSV format with the fields:
1. docId: The unique identifier for the document.
2. annotSet: Annotation set ID, assigned sequentially starting from 1 for each logical grouping of quantity relations.
3. annotType: The annotation type, one of Quantity, MeasuredEntity, MeasuredProperty, or Qualifier.
4. startOffset: The starting character offset of the annotation in the text (0-based).
5. endOffset: The ending character offset (exclusive).
6. annotId: A unique identifier for the annotation in the form T1, T2, etc., numbered sequentially within each annotSet.
7. text: The text of the annotation.
8. other: Additional information, including:For Quantity: Include unit (e.g., "kg", "ppm"), optional mods(modifiers, if any), and si (SI equivalent of the unit, if applicable). For other types: Specify relationships as {relationType: targetAnnotation} (e.g., HasQuantity: T1).

Example input:
"docId": "S0016236113008041-3153"
 "text": "The bed inventory includes 13 kg of materials with concentrations <2 ppm in specific elements."
Example output:
docId                annotSet    annotType           startOffset    endOffset    annotId    text               other
S0016236113008041-3153    1        Quantity           256           261         T1         13 kg              {"unit": "kg"}
S0016236113008041-3153    1        MeasuredEntity     239           252         T2         bed inventory      {"HasQuantity": "T1"}
S0016236113008041-3153    2        Quantity           383           389         T3         <2 ppm             {"unit": "ppm", "mods": ["IsRange"]}
S0016236113008041-3153    2        MeasuredProperty   365           379         T4         concentrations     {"HasQuantity": "T3"}
S0016236113008041-3153    2        MeasuredEntity     297           305         T5         elements           {"HasProperty": "T4"}

Chain of Thought Steps:
1. Analyze Input Data: Extract the docId and analyze the text to identify quantity relations.
2. Extract Annotations: Identify Quantity phrases (e.g., numbers and units like "13 kg", "<2 ppm"). Extract related MeasuredEntity, MeasuredProperty, and any applicable Qualifier.
3. Determine Offsets: Locate and record the startOffset and endOffset of each extracted annotation in the text.
4. Establish Relationships: Define relationships between annotations using other: Use HasQuantity to link entities to quantities. Use HasProperty to link properties to entities or quantities.
5. Validate and Organize: Ensure logical consistency between annotations and verify offsets align with the text.
6. Generate Output: Format the annotations and their relationships into a TSV table.

Input:
docId: %s
text: %s
Output:
Use CoT to explain the extraction process step by step.
Provide the final TSV output strictly following the specified format.
Do not include any explanatory statements!
"""
textpaths = ["data/trial/txt/",
            "data/train/text/"]


client = OpenAI(api_key="",
                base_url="https://api.siliconflow.cn/v1")

typemap = {"Quantity": "QUANT",
           "MeasuredEntity": "ME", 
           "MeasuredProperty": "MP", 
           "Qualifier": "QUAL"}

docIds = []
textset = {}


for fileset in textpaths:
    for fn in os.listdir(fileset):
        with open(fileset+fn) as textfile:
            text = textfile.read() #.splitlines()
            #print(fn[:-4])
            textset[fn[:-4]] = text
            docIds.append(fn[:-4])


ents = {}


for docId in docIds:
    text = textset[docId]
    
    
    ents[docId] = {}
    
    
    response = client.chat.completions.create(
            model="THUDM/chatglm3-6b",  # 填写需要调用的模型名称
            messages=[{"role": "system", "content": "You are an expert in quantity relations extraction."},
                  {"role": "user", "content": message_template % (docId, text)}])
    
    result=response.choices[0].message.content
    print(result)

    break


 、
 docId T1 T2 T3 T4 T5 text - - - - - -
 S0012821X12004384-1302 - - - - - - - - -
 S0012821X12004384-1302 1 1 - - - - {"unit": "1"}
 S0012821X12004384-1302 2 2 - - - -
 S0012821X12004384-1302 3 3 - - - {"unit": "<"}
 S0012821X12004384-1302 4 4 - - {"unit": "CIE"}
 S0012821X12004384-1302 5 5 - - - {"unit": "60 min"}
 S0012821X12004384-1302 6 6 - - {"unit": "1000"}
 S0012821X12004384-1302 7 7 - - {"unit": "5还说}
 S0012821X12004384-1302 8 8 - - {"unit": "CIE"}
 S0012821X12004384-1302 9 9 - - {"unit": "m"}
 S0012821X12004384-1302 10 10 - - {"unit": "1"}
 S0012821X12004384-1302 11 11 - - {"unit": "<"}
 S0012821X12004384-1302 12 12 - - {"unit": "CIE"}
 S0012821X12004384-1302 13 13 - - {"unit": "1"}
 S0012821X12004384-1302 14 14 - - {"unit": "<"}
 S0012821X12004384-1302 15 15 - - {"unit": "CIE"}
 S0012821X12004384-1302 16 16 - - {"unit": "1"}
 S0012821X12004384-1302 17 17 - - {"unit": "<"}
 S0012821X12004384-1302 18 18 - - {"unit": "CIE"}
 S0012821X12004384-1302 19 19 - - {"unit": "1"}
 S001